In [ ]:
# =============================================================================
# jobs/13_warning_constituency_contacts.py
# Link current EA flood warnings and alerts to constituencies and MPs.
#
# What this script does:
#   1. Reads the current state of every flood warning and alert
#      (flood_warnings_current, written by 11_flood_warnings_ingest).
#   2. Finds the constituencies each warning/alert area overlaps, using the
#      static spatial lookup (ea_area_constituency_lookup, written by 06).
#   3. Attaches the sitting MP and their contact details
#      (mp_contact_details, written by 04).
#   4. Overwrites flood_warning_constituency_contacts in full.
#
# GRAIN: one row per (flood warning or alert, constituency). A warning area that
# spans three constituencies gives three rows. Constituencies with no matching
# MP (a vacancy, or a name that does not match) are kept, with mp_matched = false
# and the MP columns NULL, so nobody is silently dropped from the table.
#
# NO SPATIAL WORK HAPPENS HERE. The geometry was done once, in 06. This notebook
# is plain Spark SQL, so it is cheap enough to run on every poll.
#
# JOIN KEYS
#   flood_warnings_current.flood_area_id = ea_area_constituency_lookup.ea_area_code
#       (the EA real-time API's floodAreaID is the same code as the area layer's
#        fws_tacode; the job fails if NO warning matches, which would mean the
#        two have stopped agreeing)
#   lookup.constituency_name ~ mp_contact_details.constituency_name
#       (the Parliament API gives no ONS code, so this joins on a normalised
#        name: lower case, "&" as "and", letters and digits only)
#
# severity_level: 1 Severe Flood Warning, 2 Flood Warning, 3 Flood Alert,
# 4 Warning no Longer in Force. is_in_force is true for 1 to 3. Level 4 rows are
# kept so a closing warning is still visible; filter on is_in_force for alerts.
#
# Exit values: "success".
# =============================================================================

import os
import sys

# Resolve the project root from this notebook's own location, so nothing
# depends on one person's workspace folder.
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

from config import (
    TBL_FLOOD_WARN_CURRENT, TBL_EA_CONST_LOOKUP, TBL_MP_CONTACTS,
    TBL_WARNING_CONTACTS, WARNING_CONTACT_MIN_PCT
)
from utils.helpers import get_spark, utc_now, table_exists, constituency_name_key


# =============================================================================
# SETUP
# =============================================================================

spark   = get_spark()
now_iso = utc_now().isoformat()

for table in (TBL_FLOOD_WARN_CURRENT, TBL_EA_CONST_LOOKUP, TBL_MP_CONTACTS):
    if not table_exists(spark, table):
        raise Exception(
            f"{table} does not exist. Run 11_flood_warnings_ingest, "
            f"06_compute_lookup and 04_refresh_mp_details first."
        )


In [ ]:
# =============================================================================
# BUILD THE TABLE
# =============================================================================

# Shared with 14: lower case, accents folded, "&" as "and", letters and digits only.
name_key = constituency_name_key


query = f"""
WITH mp AS (
    -- One MP per constituency key, so a duplicate name cannot fan rows out.
    SELECT * FROM (
        SELECT
            m.*,
            {name_key('m.constituency_name')} AS name_key,
            ROW_NUMBER() OVER (
                PARTITION BY {name_key('m.constituency_name')}
                ORDER BY m.member_id
            ) AS rn
        FROM {TBL_MP_CONTACTS} m
    ) ranked
    WHERE rn = 1
)
SELECT
    -- the warning or alert
    w.flood_area_id,
    l.ea_area_type,
    l.type_code,
    l.type_text,
    l.ea_area_name                       AS flood_area_name,
    w.severity_level,
    w.severity_label,
    (w.severity_level BETWEEN 1 AND 3)   AS is_in_force,
    w.is_tidal,
    w.message,
    w.time_raised,
    w.time_message_changed,
    w.time_severity_changed,
    w.description,
    w.ea_area_name,
    w.river_or_sea,
    w.county,
    -- the constituency
    l.constituency_id,
    l.constituency_name,
    l.intersection_pct                   AS share_of_area_in_constituency,
    -- the MP
    (mp.member_id IS NOT NULL)           AS mp_matched,
    mp.member_id                         AS mp_member_id,
    mp.name                              AS mp_name,
    mp.party                             AS mp_party,
    mp.email                             AS mp_email,
    mp.phone                             AS mp_phone,
    mp.website                           AS mp_website,
    mp.twitter_handle                    AS mp_twitter,
    -- audit
    w.ingested_at                        AS warning_ingested_at,
    '{now_iso}'                          AS computed_at
FROM {TBL_FLOOD_WARN_CURRENT} w
JOIN {TBL_EA_CONST_LOOKUP} l
  ON l.ea_area_code = w.flood_area_id
 AND l.intersection_pct >= {WARNING_CONTACT_MIN_PCT}
LEFT JOIN mp
  ON mp.name_key = {name_key('l.constituency_name')}
"""

result = spark.sql(query).cache()
n_rows = result.count()

n_warnings = spark.table(TBL_FLOOD_WARN_CURRENT).count()
n_linked   = result.select("flood_area_id").distinct().count()
print(f"Warnings and alerts in the current state:     {n_warnings}")
print(f"  linked to at least one constituency:        {n_linked}")
print(f"Rows (warning/alert x constituency):          {n_rows}")

# --- Diagnostics -------------------------------------------------------------
if n_warnings > 0 and n_linked == 0:
    # Every warning failed to match the lookup: the two code systems no longer
    # agree (or the lookup is empty). Do not publish an empty table as "fine".
    raise Exception(
        "No current warning matched ea_area_constituency_lookup on "
        "flood_area_id = ea_area_code. Check both code formats and that "
        "06_compute_lookup has run."
    )

unlinked = (
    spark.table(TBL_FLOOD_WARN_CURRENT).select("flood_area_id", "description")
    .join(result.select("flood_area_id").distinct(), "flood_area_id", "left_anti")
    .collect()
)
if unlinked:
    print(f"\nWARNING: {len(unlinked)} warning(s) have no constituency "
          f"(not in the lookup, or below WARNING_CONTACT_MIN_PCT). First 10:")
    for r in unlinked[:10]:
        print(f"  {r['flood_area_id']}  {r['description']}")

no_mp = (
    result.filter("NOT mp_matched").select("constituency_name").distinct().collect()
)
if no_mp:
    print(f"\nNOTE: {len(no_mp)} constituenc(ies) with a warning have no matching MP "
          f"(vacancy, or a name that does not match). First 10:")
    for r in no_mp[:10]:
        print(f"  {r['constituency_name']}")


In [ ]:
# =============================================================================
# WRITE TO DELTA
# Full overwrite: this table is the CURRENT picture, rebuilt every poll.
# History lives in flood_warnings_log.
# =============================================================================

(
    result.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TBL_WARNING_CONTACTS)
)

spark.sql(f"""
    COMMENT ON TABLE {TBL_WARNING_CONTACTS} IS
    'Current EA flood warnings and alerts, one row per warning/alert and constituency
     it overlaps, with the sitting MP and contact details. Rebuilt on every poll by
     13_warning_constituency_contacts from flood_warnings_current,
     ea_area_constituency_lookup and mp_contact_details. mp_matched = false marks a
     constituency with no matching MP. is_in_force is false for severity 4.'
""")

written = spark.table(TBL_WARNING_CONTACTS).count()
print(f"\nVerified: {written} rows written to {TBL_WARNING_CONTACTS}.")
result.unpersist()

dbutils.notebook.exit("success")
